Extração dos datasets

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path().cwd() / "Data-LI"
LI_ACCOUNT_DATA_PATH = DATA_DIR / "LI-Small_accounts.csv"
LI_TRANSACTIONS_DATA_PATH = DATA_DIR / "LI-Small_Trans.csv"

accounts_df = pd.read_csv(LI_ACCOUNT_DATA_PATH)
trans_df = pd.read_csv(LI_TRANSACTIONS_DATA_PATH, parse_dates=['Timestamp'])

Renaming columns with ambiguos names

In [2]:
trans_df = trans_df.rename(columns={'Account':'From Account', 'Account.1':'To Account'})
accounts_df = accounts_df.rename(columns={'Account Number': 'Account'})

Creating an universal identifier to guarantee atomicity on the dataset

In [3]:
def create_universal_id(df, bank_col, account_col):
    bank_str = df[bank_col].astype(str)
    account_str = df[account_col].astype(str)
    return bank_str + "_" + account_str

accounts_df['Universal_Account_ID'] = create_universal_id(accounts_df, 'Bank ID', 'Account')

trans_df['From_Universal_ID'] = create_universal_id(trans_df, 'From Bank', 'From Account')
trans_df['To_Universal_ID'] = create_universal_id(trans_df, 'To Bank', 'To Account')

Adjusting the volume of the transactions to a standard currency

In [4]:
FX_RATES_TO_USD = {
    'US Dollar': 1.0,
    'Euro': 1.00,
    'UK Pound': 1.15,
    'Swiss Franc': 1.03,
    'Canadian Dollar': 0.76,
    'Australian Dollar': 0.68,
    'Yen': 0.0070,            
    'Yuan': 0.144,            
    'Brazil Real': 0.193,     
    'Mexican Peso': 0.050,    
    'Rupee': 0.0125,          
    'Ruble': 0.0165,          
    'Shekel': 0.292,          
    'Saudi Riyal': 0.266,     
    'Bitcoin': 20000.0        
}

trans_df['Amount_Paid_USD'] = (
    trans_df['Amount Paid'] * trans_df['Payment Currency'].map(FX_RATES_TO_USD).fillna(1.0)
    )
trans_df['Amount_Received_USD'] = (
    trans_df['Amount Received'] * trans_df['Receiving Currency'].map(FX_RATES_TO_USD).fillna(1.0)
    )

Separating and cleaning the entity name field.

In [5]:
entity_mapping = accounts_df.set_index('Universal_Account_ID')['Entity Name']
entity_mapping_clean = entity_mapping.str.replace(r'\s*#\d+', '', regex=True)

trans_df['From_Entity_Type'] = trans_df['From_Universal_ID'].map(entity_mapping_clean)
trans_df['To_Entity_Type'] = trans_df['To_Universal_ID'].map(entity_mapping_clean)

Encoding entity name into entity type to make it passable to the model

In [ ]:
def apply_frequency_encoding(df, columns: list):
    df_encoded = df.copy()
    for col in columns:
        freq_map = df_encoded[col].value_counts(normalize=True)
        df_encoded[col] = df_encoded[col].map(freq_map)
    return df_encoded

In [7]:
entity_cols = ['From_Entity_Type', 'To_Entity_Type']
trans_df = apply_frequency_encoding(trans_df, entity_cols)

trans_df = trans_df.rename(columns={
    'From_Entity_Type': 'From_Entity_Frequency',
    'To_Entity_Type': 'To_Entity_Frequency'
})

In [8]:
def compute_rolling_stats(temp_df, group_col, window, prefix):
    grouped = temp_df.groupby(group_col)['Amount_Paid_USD']
    roll_df = grouped.rolling(window).agg(['sum', 'count']).reset_index()
    
    roll_df = roll_df.rename(columns={
        'sum': f'{prefix}_vol_{window}',
        'count': f'{prefix}_count_{window}'
    }).drop_duplicates(subset=[group_col, 'Timestamp'], keep='last')

    return roll_df

In [9]:
# Configuração inicial
trans_df = trans_df.sort_values(by='Timestamp').reset_index(drop=True)
temp_df = trans_df.set_index('Timestamp')
windows = ['1h', '24h']

for window in windows:
    roll_from = compute_rolling_stats(temp_df, 'From_Universal_ID', window, 'from')
    roll_to = compute_rolling_stats(temp_df, 'To_Universal_ID', window, 'to')
    
    roll_received_by_sender = roll_to.rename(columns={
        'To_Universal_ID': 'From_Universal_ID',
        f'to_vol_{window}': f'sender_received_vol_{window}',
        f'to_count_{window}': f'sender_received_count_{window}'
    })
    
    sender_features = roll_from.merge(
        roll_received_by_sender, 
        on=['From_Universal_ID', 'Timestamp'], 
        how='outer' 
    )
    
    trans_df = trans_df.merge(sender_features, on=['From_Universal_ID', 'Timestamp'], how='left')
    trans_df = trans_df.merge(roll_to, on=['To_Universal_ID', 'Timestamp'], how='left')
    
    cols_to_fill = [f'sender_received_vol_{window}', f'from_vol_{window}', f'from_count_{window}']
    trans_df[cols_to_fill] = trans_df[cols_to_fill].fillna(0)
    
    trans_df[f'pass_through_ratio_{window}'] = (
        trans_df[f'from_vol_{window}'] / (trans_df[f'sender_received_vol_{window}'] + 1.0)
    )
    
    trans_df[f'from_avg_ticket_{window}'] = (
        trans_df[f'from_vol_{window}'] / trans_df[f'from_count_{window}'].clip(lower=1.0)
    )

In [10]:
def calculate_rest_time(df):
    df_recebimentos = df[['Timestamp', 'To_Universal_ID']].rename(
        columns={'To_Universal_ID': 'Account', 'Timestamp': 'Last_Received_Time'}
    ).sort_values('Last_Received_Time')

    df_envios = df[['Timestamp', 'From_Universal_ID']].rename(
        columns={'From_Universal_ID': 'Account', 'Timestamp': 'Send_Time'}
    )
    df_envios['orig_index'] = df_envios.index
    df_envios = df_envios.sort_values('Send_Time')

    df_rest_time = pd.merge_asof(
        df_envios,
        df_recebimentos,
        by='Account',
        left_on='Send_Time',
        right_on='Last_Received_Time',
        direction='backward'
    )

    diff_seconds = (df_rest_time['Send_Time'] - df_rest_time['Last_Received_Time']).dt.total_seconds()
    df_rest_time['Rest_Time_Hours'] = diff_seconds / 3600.0

    return df_rest_time.set_index('orig_index')['Rest_Time_Hours'].fillna(-1.0)

In [11]:
trans_df['Ticket_Deviation_Ratio_24h'] = (
    trans_df['Amount_Paid_USD'] / (trans_df['from_avg_ticket_24h'] + 1.0)
)

trans_df['Rest_Time_Hours'] = calculate_rest_time(trans_df)

print(trans_df[['pass_through_ratio_1h', 'Ticket_Deviation_Ratio_24h', 'Rest_Time_Hours']].head())

   pass_through_ratio_1h  Ticket_Deviation_Ratio_24h  Rest_Time_Hours
0               0.999995                    0.999995              0.0
1               0.999971                    0.999971              0.0
2               0.960983                    0.960983              0.0
3               0.355670                    0.355670              0.0
4           16077.820000                    0.999938             -1.0


In [12]:
import networkx as nx
import pandas as pd

def build_temporal_graph(window_df):
    edges_df = window_df.groupby(
        ['From_Universal_ID', 'To_Universal_ID'], as_index=False
    )['Amount_Paid_USD'].sum()
    
    return nx.from_pandas_edgelist(
        df=edges_df,
        source='From_Universal_ID',
        target='To_Universal_ID',
        edge_attr='Amount_Paid_USD',
        create_using=nx.DiGraph
    )

In [13]:
def extract_node_metrics(temporal_graph, period):
    in_degrees = dict(temporal_graph.in_degree())
    out_degrees = dict(temporal_graph.out_degree())
    in_strength = dict(temporal_graph.in_degree(weight='Amount_Paid_USD'))
    out_strength = dict(temporal_graph.out_degree(weight='Amount_Paid_USD'))
    
    metrics = []
    for node in temporal_graph.nodes():
        k_in = in_degrees.get(node, 0)
        k_out = out_degrees.get(node, 0)
        w_in = in_strength.get(node, 0.0)
        w_out = out_strength.get(node, 0.0)
        
        asymmetry_ratio = (k_in - k_out) / (k_in + k_out + 1e-5)
        flow_retention_ratio = (w_in - w_out) / (w_in + w_out + 1e-5)
        
        metrics.append({
            'Universal_Account_ID': node,
            'Reference_Week': period,
            'Weekly_In_Degree': k_in,
            'Weekly_Out_Degree': k_out,
            'Degree_Asymmetry_7D': asymmetry_ratio,
            'Net_Flow_Ratio_7D': flow_retention_ratio
        })
        
    return metrics

In [14]:
time_slices = trans_df.groupby(pd.Grouper(key='Timestamp', freq='7D'))
topological_features = []

for period, window_df in time_slices:
    if not window_df.empty:
        G = build_temporal_graph(window_df)
        period_metrics = extract_node_metrics(G, period)
        
        topological_features.extend(period_metrics)

network_metrics_df = pd.DataFrame(topological_features)
print(network_metrics_df.head())

  Universal_Account_ID Reference_Week  Weekly_In_Degree  Weekly_Out_Degree  \
0          0_800060CE0     2022-09-01                 1                 20   
1          0_8015F5120     2022-09-01                 4                  4   
2      11314_800990320     2022-09-01                 3                  6   
3     118293_80A82E950     2022-09-01                 3                  1   
4         11_800062930     2022-09-01                 2                  0   

   Degree_Asymmetry_7D  Net_Flow_Ratio_7D  
0            -0.904761          -0.998593  
1             0.000000           0.812591  
2            -0.333333          -0.236377  
3             0.499999           0.987493  
4             0.999995           1.000000  


In [15]:
t_min = trans_df['Timestamp'].min()
trans_df['Reference_Week'] = t_min + pd.to_timedelta((trans_df['Timestamp'] - t_min).dt.days // 7 * 7, unit='D')

base_cols = {
    'Weekly_In_Degree': 'Weekly_In_Degree',
    'Weekly_Out_Degree': 'Weekly_Out_Degree',
    'Degree_Asymmetry_7D': 'Degree_Asymmetry_7D',
    'Net_Flow_Ratio_7D': 'Net_Flow_Ratio_7D'
}

from_rename = {k: f'From_{k}' for k in base_cols.keys()}
from_rename['Universal_Account_ID'] = 'From_Universal_ID' 
from_metrics = network_metrics_df.rename(columns=from_rename)

to_rename = {k: f'To_{k}' for k in base_cols.keys()}
to_rename['Universal_Account_ID'] = 'To_Universal_ID' 
to_metrics = network_metrics_df.rename(columns=to_rename)

trans_df = trans_df.merge(from_metrics, on=['From_Universal_ID', 'Reference_Week'], how='left')
trans_df = trans_df.merge(to_metrics, on=['To_Universal_ID', 'Reference_Week'], how='left')

cols_topologicas = list(from_rename.values())[:-1] + list(to_rename.values())[:-1]
trans_df[cols_topologicas] = trans_df[cols_topologicas].fillna(0.0)

In [20]:
aml_features = [
    # 1. Metadados e Perfis
    'Amount_Paid_USD', 
    'From_Entity_Frequency', 
    'To_Entity_Frequency',
    'Payment Format',
    
    # 2. Comportamento Temporal (Velocidade e Fuga)
    'Rest_Time_Hours',
    'pass_through_ratio_1h',
    'pass_through_ratio_24h',
    'Ticket_Deviation_Ratio_24h',
    
    # 3. Topologia de Rede (Formação de Quadrilha)
    'From_Degree_Asymmetry_7D',
    'To_Degree_Asymmetry_7D',
    'From_Net_Flow_Ratio_7D',
    'To_Net_Flow_Ratio_7D'
]

trans_df = apply_frequency_encoding(trans_df, ['Payment Format'])
X = trans_df[aml_features]
y_true = trans_df['Is Laundering']

In [21]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

iso_forest = IsolationForest(
    n_estimators=300,        
    max_samples=20000,       
    contamination=0.01,    
    random_state=42, 
    n_jobs=-1
)

pred = iso_forest.fit_predict(X)
y_pred = [1 if x == -1 else 0 for x in pred]

print("\n--- Resultados do Isolation Forest ---")
print("\nMatriz de Confusão:")
print(confusion_matrix(y_true, y_pred))

print("\nRelatório de Classificação:")
print(classification_report(y_true, y_pred, target_names=['Normal (0)', 'Lavagem (1)']))


--- Resultados do Isolation Forest ---

Matriz de Confusão:
[[6851345   69139]
 [   3463     102]]

Relatório de Classificação:
              precision    recall  f1-score   support

  Normal (0)       1.00      0.99      0.99   6920484
 Lavagem (1)       0.00      0.03      0.00      3565

    accuracy                           0.99   6924049
   macro avg       0.50      0.51      0.50   6924049
weighted avg       1.00      0.99      0.99   6924049



In [22]:
from sklearn.metrics import precision_recall_curve, auc

scores = iso_forest.decision_function(X) * -1
trans_df['Anomaly_Score'] = scores

precision, recall, thresholds = precision_recall_curve(y_true, scores)
pr_auc = auc(recall, precision)
print(f"Área sob a Curva PR (PR-AUC): {pr_auc:.4f}")

k_valores = [100, 500, 1000]
df_ordenado = trans_df.sort_values(by='Anomaly_Score', ascending=False)

print("\n--- Relatório (Top-K) ---")
for k in k_valores:
    top_k = df_ordenado.head(k)
    fraudes_reais = top_k['Is Laundering'].sum()
    precisao = fraudes_reais / k
    print(f"Investigando o Top {k}: {fraudes_reais} lavagens isoladas (Precisão: {precisao * 100:.1f}%)")

Área sob a Curva PR (PR-AUC): 0.0009

--- Relatório (Top-K) ---
Investigando o Top 100: 0 lavagens isoladas (Precisão: 0.0%)
Investigando o Top 500: 1 lavagens isoladas (Precisão: 0.2%)
Investigando o Top 1000: 3 lavagens isoladas (Precisão: 0.3%)
